In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 2: Find everything available
import os

print("=== My Drive contents ===")
for item in os.listdir("/content/drive/MyDrive"):
    print(f"  📁 {item}")

print("\n=== Shared Drives ===")
shared = "/content/drive/Shareddrives"
if os.path.exists(shared):
    for item in os.listdir(shared):
        print(f"  📁 {item}")
else:
    print("  None found")

In [ ]:
# STEP 1: Check what models exist
import os

PROJECT_ROOT = "/content/drive/MyDrive/NLP_Project"

paths = {
    "RoBERTa": f"{PROJECT_ROOT}/models/roberta_semantic/final_model",
    "DeBERTa":  f"{PROJECT_ROOT}/models/member2/final_model/deberta",
    "SpanBERT": f"{PROJECT_ROOT}/models/spanbert_cause_qa/final_model",
}

for name, path in paths.items():
    if not os.path.exists(path):
        print(f"{name}: folder does not exist")
    else:
        files = os.listdir(path)
        if not files:
            print(f"  {name}: folder exists but is EMPTY")
        else:
            has_config  = "config.json" in files
            has_weights = any(f.endswith(".bin") or f.endswith(".safetensors") for f in files)
            status = "" if (has_config and has_weights) else "⚠️ INCOMPLETE"
            print(f"{status} {name}")
            for f in files:
                print(f"     {f}")

In [29]:
# STEP 2: Test each model individually
import torch
from transformers import pipeline

PROJECT_ROOT = "/content/drive/MyDrive/NLP_Project"
device = 0 if torch.cuda.is_available() else -1

test_text = "I am so happy because I got promoted!"

In [ ]:
print("="*50)
print("TEST 1: RoBERTa")
print("="*50)
try:
    roberta = pipeline(
        "text-classification",
        model=f"{PROJECT_ROOT}/models/roberta_semantic/final_model",
        device=device
    )
    result = roberta(test_text)[0]
    print(f" Works! label={result['label']}, score={result['score']:.4f}")
except Exception as e:
    print(f" Failed: {e}")


In [ ]:
print("\n" + "="*50)
print("TEST 2: DeBERTa")
print("="*50)
try:
    deberta = pipeline(
        "text-classification",
        model=f"{PROJECT_ROOT}/models/member2/final_model/deberta",
        device=device
    )
    result = deberta(test_text)[0]
    print(f" Works! label={result['label']}, score={result['score']:.4f}")
except Exception as e:
    print(f" Failed: {e}")

In [ ]:
print("\n" + "="*50)
print("TEST 3: SpanBERT")
print("="*50)
try:
    spanbert = pipeline(
        "question-answering",
        model=f"{PROJECT_ROOT}/models/spanbert_cause_qa/final_model",
        device=device
    )
    result = spanbert(
        question="What caused the happiness?",
        context=test_text
    )
    print(f" Works! answer='{result['answer']}', score={result['score']:.4f}")
except Exception as e:
    print(f" Failed: {e}")

In [ ]:
# STEP 3: Full Pipeline with correct label mappings per model
import torch
from transformers import pipeline

PROJECT_ROOT = "/content/drive/MyDrive/NLP_Project"
device = 0 if torch.cuda.is_available() else -1

# ── Correct label maps per model ──────────────────────────────────────
ROBERTA_EMOTIONS = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']
DEBERTA_EMOTIONS = ['neutral', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']

# Load all 3 models
print("Loading models...")
roberta  = pipeline("text-classification", model=f"{PROJECT_ROOT}/models/roberta_semantic/final_model", device=device)
deberta  = pipeline("text-classification", model=f"{PROJECT_ROOT}/models/member2/final_model/deberta", device=device)
spanbert = pipeline("question-answering",  model=f"{PROJECT_ROOT}/models/spanbert_cause_qa/final_model", device=device)
print("✅ All models loaded!\n")


In [ ]:

NEUTRAL_THRESHOLD = 0.60  # if neither model is confident enough → neutral

def get_emotion(text):
    rob = roberta(text)[0]
    deb = deberta(text)[0]

    rob_emotion = ROBERTA_EMOTIONS[int(rob['label'].split('_')[1])]
    deb_emotion = DEBERTA_EMOTIONS[int(deb['label'].split('_')[1])]

    # If both models agree it's non-neutral but confidence is low → call it neutral
    if rob['score'] >= deb['score']:
        emotion, conf, source = rob_emotion, round(rob['score'], 4), "RoBERTa"
    else:
        emotion, conf, source = deb_emotion, round(deb['score'], 4), "DeBERTa"

    # Override to neutral if:
    # 1. Both models disagree with each other
    # 2. OR winning confidence is below threshold
    if rob_emotion != deb_emotion and conf < NEUTRAL_THRESHOLD:
        return "neutral", conf, "ensemble"

    return emotion, conf, source

def predict(current_text, context_history=None):
    history_str = " | ".join(context_history) if context_history else ""
    full_context = f"{history_str} | {current_text}" if history_str else current_text

    emotion, emo_conf, source = get_emotion(current_text)

    if emotion == 'neutral':
        return {
            "emotion": "neutral",
            "cause": "N/A",
            "emo_confidence": emo_conf,
            "expert": source
        }

    cause = spanbert(question=f"What caused the {emotion}?", context=full_context)

    return {
        "emotion":          emotion,
        "cause":            cause['answer'],
        "emo_confidence":   emo_conf,
        "cause_confidence": round(cause['score'], 4),
        "expert":           source
    }

def predict_v3(current_text, context_history=None):
    history_str  = " | ".join(context_history) if context_history else ""
    full_context = f"{history_str} | {current_text}" if history_str else current_text

    emotion, emo_conf, source = get_emotion_v2(current_text, context_history)

    if emotion == 'neutral':
        return {
            "emotion":        "neutral",
            "cause":          "N/A",
            "emo_confidence": emo_conf,
            "expert":         source
        }

    cause = spanbert(question=f"What caused the {emotion}?", context=full_context)

    return {
        "emotion":          emotion,
        "cause":            cause['answer'],
        "emo_confidence":   emo_conf,
        "cause_confidence": round(cause['score'], 4),
        "expert":           source
    }

# ── Tests ─────────────────────────────────────────────────────────────
tests = [
    ("I am so happy because I got promoted!", None),
    ("[Rachel]: I'm just so happy!", ["[Ross]: I think I'm in love with you.", "[Rachel]: Oh my god, really?"]),
    ("I went to the store today.", None),
    ("She was devastated because her dog passed away.", None),
]

for text, context in tests:
    print("="*50)
    print(f"Input  : '{text}'")
    r = predict(text, context)
    print(f"Emotion: {r['emotion']} ({r['emo_confidence']}) via {r['expert']}")
    print(f"Cause  : '{r.get('cause', 'N/A')}'")

In [ ]:
# ── IMPROVED get_emotion for short utterances ─────────────────────────

def get_emotion_v2(text, context_history=None):
    word_count = len(text.replace('[', '').replace(']', '').split())

    # Build enriched version with context
    if context_history:
        context_str   = " | ".join(context_history[-2:])
        enriched_text = f"{context_str} | {text}"
    else:
        enriched_text = text

    rob = roberta(text)[0]
    deb = deberta(text)[0]

    rob_emotion = ROBERTA_EMOTIONS[int(rob['label'].split('_')[1])]
    deb_emotion = DEBERTA_EMOTIONS[int(deb['label'].split('_')[1])]

    # For short utterances (≤ 6 words):
    # RoBERTa tends to confidently say neutral even when wrong
    # DeBERTa without context is better calibrated for these
    if word_count <= 6:
        # Only trust RoBERTa if it says non-neutral
        if rob_emotion != 'neutral':
            return rob_emotion, round(rob['score'], 4), "RoBERTa"
        # Otherwise trust DeBERTa (even if low confidence)
        if deb_emotion != 'neutral':
            return deb_emotion, round(deb['score'], 4), "DeBERTa*short"
        # Both say neutral → use enriched context as tiebreaker
        rob_enr = roberta(enriched_text)[0]
        deb_enr = deberta(enriched_text)[0]
        rob_enr_emo = ROBERTA_EMOTIONS[int(rob_enr['label'].split('_')[1])]
        deb_enr_emo = DEBERTA_EMOTIONS[int(deb_enr['label'].split('_')[1])]
        if deb_enr_emo != 'neutral':
            return deb_enr_emo, round(deb_enr['score'], 4), "DeBERTa*context"
        if rob_enr_emo != 'neutral':
            return rob_enr_emo, round(rob_enr['score'], 4), "RoBERTa*context"
        return 'neutral', round(deb['score'], 4), "Both"

    # For normal length utterances: higher confidence wins
    if rob['score'] >= deb['score']:
        return rob_emotion, round(rob['score'], 4), "RoBERTa"
    else:
        return deb_emotion, round(deb['score'], 4), "DeBERTa"


def predict_v3(current_text, context_history=None):
    history_str  = " | ".join(context_history) if context_history else ""
    full_context = f"{history_str} | {current_text}" if history_str else current_text

    emotion, emo_conf, source = get_emotion_v2(current_text, context_history)

    if emotion == 'neutral':
        return {
            "emotion":        "neutral",
            "cause":          "N/A",
            "emo_confidence": emo_conf,
            "expert":         source
        }

    cause = spanbert(question=f"What caused the {emotion}?", context=full_context)

    return {
        "emotion":          emotion,
        "cause":            cause['answer'],
        "emo_confidence":   emo_conf,
        "cause_confidence": round(cause['score'], 4),
        "expert":           source
    }

# ── Quick test on all 3 failing examples ──────────────────────────────
failing = [
    ("[Chandler]: That is right .",
     ["[Chandler]: Alright , so I am back in high school",
      "[All]: Oh , yeah . Had that dream .",
      "[Chandler]: Then I look down , and I realize there is a phone ... there .",
      "[Joey]: Instead of ... ?"]),
    ("[Ross]: Oh .",
     ["[Rachel]: I always figured you just thought I was Monica geeky older brother .",
      "[Ross]: I did ."]),
    ("[Rachel]: Yeah , maybe ...",
     ["[Someone]: do you think it would be okay if I asked you out ?"]),
]

print("Comparing predict_v2 vs predict_v3 on failing examples:\n")
for text, ctx in failing:
    r_v2 = predict(text, ctx)
    r_v3 = predict_v3(text, ctx)
    #improved =  if r_v3['emotion'] != 'neutral' else
    print(f"Text : '{text}'")
    print(f"  v2 : {r_v2['emotion']:<12} | v3: {r_v3['emotion']:<12}")
    print(f"  via: {r_v3.get('expert','')}")
    print()

In [ ]:
# Complex test cases
complex_tests = [
    # 1. Sarcasm
    ("Oh great, another Monday. Just what I needed.",
     None),

    # 2. Cause is far back in context
    ("[Monica]: I got into culinary school!",
     ["[Monica]: I applied to the top culinary school in New York last month.",
      "[Rachel]: Oh you've been waiting so long for this!",
      "[Monica]: The letter finally came today..."]),

    # 3. Multiple emotions in one sentence
    ("I'm happy for her but I'm also heartbroken that she's leaving.",
     None),

    # 4. Implicit cause — not directly stated
    ("He couldn't stop crying at the funeral.",
     None),

    # 5. Anger with long context
    ("[Ross]: You had no right to do that!",
     ["[Rachel]: I read your journal Ross.",
      "[Ross]: That was private!",
      "[Rachel]: I just wanted to understand you better."]),

    # 6. Fear
    ("I can't sleep, I keep hearing noises outside my window.",
     None),

    # 7. Cause is in a different speaker's line
    ("[Chandler]: I can't believe she said yes!",
     ["[Joey]: Dude did you propose to Monica?",
      "[Chandler]: I got down on one knee and everything.",
      "[Joey]: What did she say??"]),

    # 8. Disgust
    ("That is absolutely revolting, I can't even look at it.",
     None),
]

print("COMPLEX TEST CASES")
for i, (text, context) in enumerate(complex_tests, 1):
    print(f"\n{'='*55}")
    print(f"Test {i}")
    if context:
        print(f"Context:")
        for c in context:
            print(f"  {c}")
    print(f"Input  : '{text}'")
    r = predict_v3(text, context)
    print(f"Emotion: {r['emotion']} ({r['emo_confidence']}) via {r['expert']}")
    print(f"Cause  : '{r.get('cause', 'N/A')}' ", end="")
    if r.get('cause_confidence'):
        print(f"({r['cause_confidence']})")
    else:
        print()

In [ ]:
# Detailed output: both models' predictions + both SpanBERT queries

complex_tests = [
    ("Oh great, another Monday. Just what I needed.", None),
    ("[Monica]: I got into culinary school!",
     ["[Monica]: I applied to the top culinary school in New York last month.",
      "[Rachel]: Oh you've been waiting so long for this!",
      "[Monica]: The letter finally came today..."]),
    ("I'm happy for her but I'm also heartbroken that she's leaving.", None),
    ("He couldn't stop crying at the funeral.", None),
    ("[Ross]: You had no right to do that!",
     ["[Rachel]: I read your journal Ross.",
      "[Ross]: That was private!",
      "[Rachel]: I just wanted to understand you better."]),
    ("I can't sleep, I keep hearing noises outside my window.", None),
    ("[Chandler]: I can't believe she said yes!",
     ["[Joey]: Dude did you propose to Monica?",
      "[Chandler]: I got down on one knee and everything.",
      "[Joey]: What did she say??"]),
    ("That is absolutely revolting, I can't even look at it.", None),
]

for i, (text, context) in enumerate(complex_tests, 1):
    print(f"\n{'='*60}")
    print(f"Test {i}")
    if context:
        print(f"Context:")
        for c in context:
            print(f"  {c}")
    print(f"Input: '{text}'")

    history_str = " | ".join(context) if context else ""
    full_context = f"{history_str} | {text}" if history_str else text

    # ── Both emotion models independently ────────────────────────────
    rob_raw = roberta(text)[0]
    deb_raw = deberta(text)[0]

    rob_emotion = ROBERTA_EMOTIONS[int(rob_raw['label'].split('_')[1])]
    deb_emotion = DEBERTA_EMOTIONS[int(deb_raw['label'].split('_')[1])]

    print(f"\n   RoBERTa → emotion: {rob_emotion:<10} confidence: {rob_raw['score']:.4f}")
    print(f"   DeBERTa → emotion: {deb_emotion:<10} confidence: {deb_raw['score']:.4f}")

    if rob_emotion == deb_emotion:
        print(f"  Both AGREE: {rob_emotion}")
    else:
        winner = "RoBERTa" if rob_raw['score'] >= deb_raw['score'] else "DeBERTa"
        print(f"  DISAGREE → {winner} wins (higher confidence)")

    # ── SpanBERT with RoBERTa's emotion ──────────────────────────────
    if rob_emotion != 'neutral':
        span1 = spanbert(question=f"What caused the {rob_emotion}?", context=full_context)
        print(f"\n   Span1 (RoBERTa emotion '{rob_emotion}'):")
        print(f"     answer: '{span1['answer']}' (score: {span1['score']:.4f})")
    else:
        print(f"\n  Span1 (RoBERTa): neutral → skipped")

    # ── SpanBERT with DeBERTa's emotion ──────────────────────────────
    if deb_emotion != 'neutral':
        span2 = spanbert(question=f"What caused the {deb_emotion}?", context=full_context)
        print(f"\n  Span2 (DeBERTa emotion '{deb_emotion}'):")
        print(f"     answer: '{span2['answer']}' (score: {span2['score']:.4f})")
    else:
        print(f"\n  Span2 (DeBERTa): neutral → skipped")

    # ── Final decision ────────────────────────────────────────────────
    final_emotion = rob_emotion if rob_raw['score'] >= deb_raw['score'] else deb_emotion
    final_span    = span1 if rob_raw['score'] >= deb_raw['score'] else span2
    if final_emotion != 'neutral':
        print(f"\n  FINAL → emotion: {final_emotion}, cause: '{final_span['answer']}'")
    else:
        print(f"\n  FINAL → emotion: neutral, cause: N/A")

In [ ]:
# ── SemEval-2024 Task 3 Official Evaluation ──────────────────────────

import json, re
from collections import defaultdict

# ── Token-level overlap (for Proportional F1) ─────────────────────────
def token_overlap(pred_span, gold_span):
    pred_tokens = set(re.findall(r'\w+', pred_span.lower()))
    gold_tokens = set(re.findall(r'\w+', gold_span.lower()))
    if not gold_tokens:
        return 1.0 if not pred_tokens else 0.0
    return len(pred_tokens & gold_tokens) / len(gold_tokens)

# ── Per-emotion accumulators ───────────────────────────────────────────
EMOTIONS = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
ALL_EMOTIONS = EMOTIONS + ['neutral']

strict = {e: {'tp': 0, 'fp': 0, 'fn': 0} for e in ALL_EMOTIONS}
prop   = {e: {'overlap_sum': 0.0, 'pred_count': 0, 'gold_count': 0} for e in ALL_EMOTIONS}

emo_correct = 0
emo_total   = 0
per_emo_acc = defaultdict(lambda: {'correct': 0, 'total': 0})
errors      = []

# ── Load data ─────────────────────────────────────────────────────────
data_path = f'{PROJECT_ROOT}/data/Subtask_1_test.json'
with open(data_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data)} conversations\n")

# ── Main evaluation loop ──────────────────────────────────────────────
for conv_idx, conversation in enumerate(raw_data):
    utterances          = conversation.get('conversation', [])
    emotion_cause_pairs = conversation.get('emotion-cause_pairs', [])

    utt_map = {u['utterance_ID']: u for u in utterances}

    for i, utt in enumerate(utterances):
        utt['_context'] = [
            f"[{utterances[j].get('speaker','')}]: {utterances[j].get('text','')}"
            for j in range(max(0, i - 3), i)
        ]

    # Parse gold pairs into structured tuples
    gold_tuples = []
    for pair in emotion_cause_pairs:
        emotion_part = pair[0]
        cause_part   = pair[1]
        ep = emotion_part.split('_', 1)
        cp = cause_part.split('_', 1)
        if len(ep) != 2 or len(cp) != 2:
            continue
        try:
            emotion_utt_id = int(ep[0])
            cause_utt_id   = int(cp[0])
        except ValueError:
            continue
        gold_emotion = ep[1].lower().strip()
        gold_span    = cp[1].strip()
        if not gold_span:
            continue
        gold_tuples.append({
            'emotion_utt_id': emotion_utt_id,
            'emotion':        gold_emotion,
            'cause_utt_id':   cause_utt_id,
            'span':           gold_span,
        })

    # Group gold tuples by emotion utterance
    by_emo_utt = defaultdict(list)
    for gt in gold_tuples:
        by_emo_utt[gt['emotion_utt_id']].append(gt)

    for emotion_utt_id, gold_list in by_emo_utt.items():
        if emotion_utt_id not in utt_map:
            continue

        emotion_utt     = utt_map[emotion_utt_id]
        emotion_text    = f"[{emotion_utt.get('speaker','')}]: {emotion_utt.get('text','')}"
        context_history = emotion_utt.get('_context', [])
        gold_emotion    = gold_list[0]['emotion']

        emo_total += 1
        per_emo_acc[gold_emotion]['total'] += 1

        # Count all gold tuples for recall denominators
        for gt in gold_list:
            prop[gt['emotion']]['gold_count']  += 1
            strict[gt['emotion']]['fn']        += 1   # will undo if matched

        # Run pipeline
        try:
            pred = predict(emotion_text, context_history)
        except Exception:
            continue

        pred_emotion = pred['emotion']
        pred_span    = pred.get('cause', '').strip()
        if pred_span == 'N/A':
            pred_span = ''

        # Emotion accuracy
        if pred_emotion == gold_emotion:
            emo_correct += 1
            per_emo_acc[gold_emotion]['correct'] += 1
        elif len(errors) < 5:
            errors.append({
                'text':       emotion_text,
                'gold':       gold_emotion,
                'pred':       pred_emotion,
                'gold_spans': [g['span'] for g in gold_list],
                'pred_span':  pred_span,
            })

        # ── Strict F1 ─────────────────────────────────────────────────
        # Match if: pred_emotion == gold_emotion AND span exact match
        matched_gi = None
        if pred_emotion != 'neutral' and pred_span:
            for gi, gt in enumerate(gold_list):
                if (pred_emotion == gt['emotion'] and
                        pred_span.lower() == gt['span'].lower()):
                    matched_gi = gi
                    break

        if matched_gi is not None:
            strict[pred_emotion]['tp'] += 1
            strict[pred_emotion]['fn'] -= 1   # undo the pre-counted FN
        elif pred_emotion != 'neutral' and pred_span:
            strict[pred_emotion]['fp'] += 1

        # ── Proportional F1 ───────────────────────────────────────────
        # Emotion must match; span gets best token-overlap credit
        if pred_emotion != 'neutral' and pred_span:
            prop[pred_emotion]['pred_count'] += 1
            best_overlap = 0.0
            for gt in gold_list:
                if pred_emotion == gt['emotion']:
                    ov = token_overlap(pred_span, gt['span'])
                    best_overlap = max(best_overlap, ov)
            prop[pred_emotion]['overlap_sum'] += best_overlap

    if (conv_idx + 1) % 200 == 0:
        print(f"  Processed {conv_idx+1}/{len(raw_data)} conversations...")

# ── Compute F1 scores ─────────────────────────────────────────────────
def get_strict_f1(e):
    tp = strict[e]['tp']
    fp = strict[e]['fp']
    fn = strict[e]['fn']
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2*p*r / (p+r)  if (p + r)  > 0 else 0.0
    return p, r, f1

def get_prop_f1(e):
    pc   = prop[e]['pred_count']
    gc   = prop[e]['gold_count']
    osum = prop[e]['overlap_sum']
    p    = osum / pc if pc > 0 else 0.0
    r    = osum / gc if gc > 0 else 0.0
    f1   = 2*p*r / (p+r) if (p+r) > 0 else 0.0
    return p, r, f1

total_gold     = sum(prop[e]['gold_count'] for e in EMOTIONS)
wavg_strict_f1 = sum(get_strict_f1(e)[2] * prop[e]['gold_count'] for e in EMOTIONS) / total_gold if total_gold else 0.0
wavg_prop_f1   = sum(get_prop_f1(e)[2]   * prop[e]['gold_count'] for e in EMOTIONS) / total_gold if total_gold else 0.0

# ── Print Full Results ────────────────────────────────────────────────
SEP = "=" * 70

print("\n" + SEP)
print("  SemEval-2024 Task 3 — Official Evaluation Results")
print(SEP)

# 1. Emotion Detection
print(f"\n{'='*70}")
print(f"EMOTION DETECTION ACCURACY")
print(f"  Overall : {emo_correct}/{emo_total} = {100*emo_correct/max(emo_total,1):.2f}%")
print(f"\n  {'Emotion':<12} {'Correct':>8} {'Total':>8} {'Acc%':>8}")
print(f"  {'-'*40}")
for emo in ALL_EMOTIONS:
    s = per_emo_acc[emo]
    if s['total'] > 0:
        acc = 100 * s['correct'] / s['total']
        bar = chr(0x2588) * int(acc / 5)
        print(f"  {emo:<12} {s['correct']:>8} {s['total']:>8} {acc:>7.2f}%  {bar}")

# 2. Strict F1
print(f"\n{'='*70}")
print(f"STRICT EVALUATION  (exact span + exact emotion match)")
print(f"  w-avg. Strict F1 = {wavg_strict_f1:.4f}  <- PRIMARY RANKING METRIC")
print(f"\n  {'Emotion':<12} {'Prec':>8} {'Rec':>8} {'F1':>8} {'#Gold':>8}")
print(f"  {'-'*50}")
for emo in EMOTIONS:
    p, r, f1 = get_strict_f1(emo)
    gc = prop[emo]['gold_count']
    print(f"  {emo:<12} {p:>8.4f} {r:>8.4f} {f1:>8.4f} {gc:>8}")
print(f"\n  {'w-avg':<12} {'':>8} {'':>8} {wavg_strict_f1:>8.4f} {total_gold:>8}")

# 3. Proportional F1
print(f"\n{'='*70}")
print(f"PROPORTIONAL EVALUATION  (token-overlap span credit)")
print(f"  w-avg. Proportional F1 = {wavg_prop_f1:.4f}")
print(f"\n  {'Emotion':<12} {'Prec':>8} {'Rec':>8} {'F1':>8} {'#Gold':>8}")
print(f"  {'-'*50}")
for emo in EMOTIONS:
    p, r, f1 = get_prop_f1(emo)
    gc = prop[emo]['gold_count']
    print(f"  {emo:<12} {p:>8.4f} {r:>8.4f} {f1:>8.4f} {gc:>8}")
print(f"\n  {'w-avg':<12} {'':>8} {'':>8} {wavg_prop_f1:>8.4f} {total_gold:>8}")

# 4. Summary
print(f"\n{'='*70}")
print(f"SUMMARY TABLE")
print(f"  {'Metric':<40} {'Score':>10}")
print(f"  {'-'*52}")
print(f"  {'Emotion Accuracy':<40} {100*emo_correct/max(emo_total,1):>9.2f}%")
print(f"  {'w-avg. Strict F1  (PRIMARY METRIC)':<40} {wavg_strict_f1:>10.4f}")
print(f"  {'w-avg. Proportional F1':<40} {wavg_prop_f1:>10.4f}")
print(f"  {'Total gold tuples evaluated':<40} {total_gold:>10}")

# 5. Sample errors
if errors:
    print(f"\n{'='*70}")
    print(f"SAMPLE EMOTION ERRORS (first 5)")
    for r in errors:
        print(f"\n  Text       : '{r['text']}'")
        print(f"  Gold       : {r['gold']}  ->  Predicted: {r['pred']}")
        print(f"  Gold spans : {r['gold_spans']}")
        print(f"  Pred span  : '{r['pred_span']}'")

print("\n" + SEP)
